# Семинар 8. Метрики качества и интерпретация моделей

В этом семинаре мы разберем:
- Метрики классификации (confusion matrix, precision/recall/F1, ROC-AUC, PR-AUC, подбор порога)
- Метрики регрессии (MSE, MAE, MAPE, R2)
- Интерпретация моделей (SHAP, LIME, Partial Dependence Plots)
- Калибровка вероятностей

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q catboost shap lime

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from catboost import CatBoostClassifier

np.random.seed(42)

## 1. Метрики классификации

### Данные

Возьмем несбалансированный датасет: 95% класса 0, 5% класса 1. Это типичная ситуация в реальных задачах (fraud detection, диагностика заболеваний).

In [ ]:
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_classes=2, weights=[0.95, 0.05], random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Train: {len(y_train)} samples, class 1: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Test:  {len(y_test)} samples, class 1: {y_test.sum()} ({y_test.mean():.1%})")

# Обучим две модели для сравнения
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

### 1.1 Confusion Matrix

Confusion matrix показывает все 4 типа предсказаний:
- **TP** (True Positive): модель сказала 1, правда 1
- **FP** (False Positive): модель сказала 1, правда 0 (ложная тревога)
- **FN** (False Negative): модель сказала 0, правда 1 (пропуск)
- **TN** (True Negative): модель сказала 0, правда 0

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, name in zip(axes, [lr, rf], ['LogisticRegression', 'RandomForest']):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

### 1.2 Accuracy, Precision, Recall, F1

- **Accuracy** = (TP+TN) / (TP+TN+FP+FN) - доля правильных ответов. На несбалансированных данных бесполезна: "всегда предсказываем 0" дает accuracy 95%.
- **Precision** = TP / (TP+FP) - из тех, кого мы назвали положительными, сколько действительно положительные. Важна, когда FP дорого (спам-фильтр: не хотим терять письма).
- **Recall** = TP / (TP+FN) - из всех реально положительных, сколько мы нашли. Важна, когда FN дорого (диагностика рака: не хотим пропустить).
- **F1** = 2 * Precision * Recall / (Precision + Recall) - гармоническое среднее.

In [ ]:
for model, name in [(lr, 'LogReg'), (rf, 'RF')]:
    preds = model.predict(X_test)
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy_score(y_test, preds):.4f}")
    print(f"  Precision: {precision_score(y_test, preds):.4f}")
    print(f"  Recall:    {recall_score(y_test, preds):.4f}")
    print(f"  F1:        {f1_score(y_test, preds):.4f}")

# "Тупая" модель
dummy_preds = np.zeros_like(y_test)
print(f"\nDummy (all zeros):")
print(f"  Accuracy:  {accuracy_score(y_test, dummy_preds):.4f}  <-- высокая, но бесполезная")
print(f"  Recall:    {recall_score(y_test, dummy_preds):.4f}  <-- ноль, ничего не нашли")

### 1.3 ROC curve и AUC

ROC (Receiver Operating Characteristic) строит зависимость TPR (True Positive Rate = Recall) от FPR (False Positive Rate) при разных порогах классификации.

- Идеальная модель: AUC = 1.0 (кривая проходит через верхний левый угол)
- Случайная модель: AUC = 0.5 (диагональ)
- AUC не зависит от порога - оценивает модель "в целом"

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for model, name, color in [(lr, 'LogReg', 'blue'), (rf, 'RF', 'green')]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True)
ax.set_aspect('equal')
plt.show()

### 1.4 PR curve (Precision-Recall)

На несбалансированных данных ROC-AUC может быть обманчиво высокой. PR curve лучше показывает качество модели на редком классе.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for model, name, color in [(lr, 'LogReg', 'blue'), (rf, 'RF', 'green')]:
    proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(recall, precision, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')

# Baseline: доля положительного класса
baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True)
plt.show()

### 1.5 Подбор порога классификации

По умолчанию порог = 0.5. Но оптимальный порог зависит от задачи:
- Хотим максимизировать F1 -> ищем порог с лучшим F1
- Хотим recall >= 0.9 -> двигаем порог вниз

In [ ]:
proba_rf = rf.predict_proba(X_test)[:, 1]
thresholds = np.linspace(0.01, 0.99, 200)

precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds_t = (proba_rf >= t).astype(int)
    if preds_t.sum() == 0:
        precisions.append(0)
        recalls.append(0)
        f1s.append(0)
        continue
    precisions.append(precision_score(y_test, preds_t, zero_division=0))
    recalls.append(recall_score(y_test, preds_t))
    f1s.append(f1_score(y_test, preds_t))

best_idx = np.argmax(f1s)
best_threshold = thresholds[best_idx]

plt.figure(figsize=(12, 6))
plt.plot(thresholds, precisions, label='Precision', linewidth=2)
plt.plot(thresholds, recalls, label='Recall', linewidth=2)
plt.plot(thresholds, f1s, label='F1', linewidth=2)
plt.axvline(x=0.5, color='gray', linestyle=':', label='Default threshold (0.5)')
plt.axvline(x=best_threshold, color='red', linestyle='--',
            label=f'Best F1 threshold ({best_threshold:.2f}), F1={f1s[best_idx]:.3f}')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision / Recall / F1 vs Threshold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()

## 2. Метрики регрессии

In [ ]:
X_reg, y_reg = make_regression(n_samples=500, n_features=10, n_informative=5, noise=20.0, random_state=42)
X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

gb_reg = GradientBoostingRegressor(n_estimators=100, random_state=42).fit(X_reg_tr, y_reg_tr)
preds_reg = gb_reg.predict(X_reg_te)

mse = mean_squared_error(y_reg_te, preds_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_reg_te, preds_reg)
r2 = r2_score(y_reg_te, preds_reg)
# MAPE
mask = y_reg_te != 0
mape = np.mean(np.abs((y_reg_te[mask] - preds_reg[mask]) / y_reg_te[mask])) * 100

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}  (в тех же единицах, что и таргет)")
print(f"MAE:  {mae:.2f}  (устойчива к выбросам)")
print(f"MAPE: {mape:.1f}%  (относительная ошибка)")
print(f"R2:   {r2:.4f}  (доля объясненной дисперсии, 1.0 = идеал)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicted vs Actual
axes[0].scatter(y_reg_te, preds_reg, alpha=0.5, s=20)
lims = [min(y_reg_te.min(), preds_reg.min()), max(y_reg_te.max(), preds_reg.max())]
axes[0].plot(lims, lims, 'r--', label='Ideal')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predicted vs Actual (R2={r2:.3f})')
axes[0].legend()
axes[0].grid(True)

# Residuals
residuals = y_reg_te - preds_reg
axes[1].scatter(preds_reg, residuals, alpha=0.5, s=20)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual plot')
axes[1].grid(True)

plt.tight_layout()
plt.show()

| Метрика | Формула | Когда использовать |
|---|---|---|
| MSE | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Дефолт, чувствительна к выбросам |
| RMSE | $\sqrt{MSE}$ | Те же единицы, что и таргет |
| MAE | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Устойчива к выбросам |
| MAPE | $\frac{100}{n}\sum\frac{|y_i - \hat{y}_i|}{|y_i|}$ | Относительная ошибка (не работает при $y=0$) |
| R2 | $1 - \frac{SS_{res}}{SS_{tot}}$ | Доля объясненной дисперсии |

## 3. Интерпретация моделей

### Данные

Используем Breast Cancer Wisconsin - реальный медицинский датасет. Здесь интерпретируемость критична.

In [ ]:
data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_bc = data.target  # 0=malignant, 1=benign

X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42)

cb = CatBoostClassifier(iterations=200, depth=4, verbose=False, random_state=42)
cb.fit(X_bc_tr, y_bc_tr)
print(f"Test accuracy: {cb.score(X_bc_te, y_bc_te):.4f}")

### 3.1 SHAP values

SHAP (SHapley Additive exPlanations) - метод из теории игр, оценивающий вклад каждого признака в конкретное предсказание. Сумма SHAP values = разница между предсказанием модели и средним предсказанием.

In [ ]:
import shap

explainer = shap.TreeExplainer(cb)
shap_values = explainer.shap_values(X_bc_te)

In [ ]:
# Summary plot: глобальная важность + направление влияния
shap.summary_plot(shap_values, X_bc_te, max_display=15)

In [ ]:
# Waterfall: объяснение одного предсказания
idx = 0
print(f"Sample {idx}: true={y_bc_te.iloc[idx] if hasattr(y_bc_te, 'iloc') else y_bc_te[idx]}, "
      f"predicted={cb.predict(X_bc_te)[idx]}")
shap.waterfall_plot(shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_bc_te.iloc[idx],
    feature_names=X_bc_te.columns.tolist(),
))

In [ ]:
# Dependence plot: как один признак влияет на предсказание
shap.dependence_plot('worst radius', shap_values, X_bc_te)

### 3.2 LIME

LIME (Local Interpretable Model-agnostic Explanations) объясняет предсказание для конкретного объекта: строит простую линейную модель в окрестности этого объекта.

In [ ]:
import lime
import lime.lime_tabular

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_bc_tr.values, feature_names=X_bc_tr.columns.tolist(),
    class_names=['malignant', 'benign'], mode='classification',
)

# Объяснение для одного объекта
idx = 0
exp = lime_explainer.explain_instance(
    X_bc_te.values[idx], cb.predict_proba, num_features=10,
)
exp.as_pyplot_figure()
plt.title(f'LIME explanation (sample {idx})', fontsize=14)
plt.tight_layout()
plt.show()

### 3.3 Partial Dependence Plots (PDP)

PDP показывает средний эффект одного или двух признаков на предсказание модели, усредненный по всем остальным признакам.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Переобучим sklearn модель для совместимости с PDP
rf_bc = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_bc_tr, y_bc_tr)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
features_to_plot = ['worst radius', 'mean concave points', 'worst texture']
for ax, feat in zip(axes, features_to_plot):
    PartialDependenceDisplay.from_estimator(rf_bc, X_bc_tr, [feat], ax=ax)
    ax.set_title(feat)
    ax.grid(True)
plt.suptitle('Partial Dependence Plots', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 2D PDP: взаимодействие двух признаков
fig, ax = plt.subplots(figsize=(10, 8))
PartialDependenceDisplay.from_estimator(
    rf_bc, X_bc_tr, [('worst radius', 'mean concave points')], ax=ax,
)
ax.set_title('2D Partial Dependence: worst radius vs mean concave points')
plt.tight_layout()
plt.show()

## 4. Калибровка вероятностей

Когда модель предсказывает probability=0.8, действительно ли 80% таких объектов положительные? Если да - модель калибрована. Если нет - нужна калибровка.

Калибровка важна, когда вероятности используются для принятия решений (а не только для ранжирования).

In [ ]:
# Используем более крупный датасет для надежности
X_cal, y_cal = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    n_classes=2, random_state=42,
)
X_cal_tr, X_cal_te, y_cal_tr, y_cal_te = train_test_split(X_cal, y_cal, test_size=0.3, random_state=42)

# Модели
lr_cal = LogisticRegression(max_iter=1000, random_state=42).fit(X_cal_tr, y_cal_tr)
rf_cal = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_cal_tr, y_cal_tr)
cb_cal = CatBoostClassifier(iterations=200, verbose=False, random_state=42).fit(X_cal_tr, y_cal_tr)

# Калиброванная версия RF
rf_calibrated = CalibratedClassifierCV(rf_cal, cv=5, method='isotonic').fit(X_cal_tr, y_cal_tr)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models = [
    (lr_cal, 'LogReg', 'blue'),
    (rf_cal, 'RF (uncalibrated)', 'green'),
    (rf_calibrated, 'RF (calibrated)', 'red'),
    (cb_cal, 'CatBoost', 'orange'),
]

for model, name, color in models:
    proba = model.predict_proba(X_cal_te)[:, 1]
    fraction_pos, mean_pred = calibration_curve(y_cal_te, proba, n_bins=10)
    axes[0].plot(mean_pred, fraction_pos, 'o-', color=color, label=name, linewidth=2)
    axes[1].hist(proba, bins=20, alpha=0.4, color=color, label=name)

axes[0].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].set_title('Reliability diagram (calibration curve)')
axes[0].legend(fontsize=10)
axes[0].grid(True)

axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of predicted probabilities')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

## Итоги

### Метрики классификации
- **Accuracy** - бесполезна на несбалансированных данных
- **Precision/Recall** - зависят от порога, выбирайте в зависимости от цены ошибки
- **F1** - баланс precision и recall
- **ROC-AUC** - общая оценка модели, не зависит от порога
- **PR-AUC** - лучше ROC-AUC на несбалансированных данных

### Интерпретация
- **SHAP** - глобальная и локальная интерпретация, теоретически обоснован
- **LIME** - локальная интерпретация, model-agnostic
- **PDP** - средний эффект признака на предсказание

### Калибровка
- LogReg обычно хорошо калиброван
- RF и бустинги часто нуждаются в калибровке
- `CalibratedClassifierCV` (isotonic/Platt) исправляет калибровку